# Подготовка модели для рекомендации тарифных планов
<a class="anchor" id="content"></a>

<a class='anchor' id='content'></a>
## Содержание

1. [Описание проекта](#desc)
2. [Получение данных](#get)
3. [Ход исследования](#research)
4. [Результаты исследования](#result)
5. [Чек-лист](#check)

<a class='anchor' id='desc'></a>
## 1. Описание проекта

Оператор мобильной связи «Мегалайн» выяснил: многие клиенты пользуются архивными тарифами. Они хотят построить систему, способную проанализировать поведение клиентов и предложить пользователям новый тариф: «Смарт» или «Ультра».

В нашем распоряжении данные о поведении клиентов, которые уже перешли на эти тарифы. Нужно построить модель для задачи классификации, которая выберет подходящий тариф. Предобработка данных ранее была выполнена.

#### Цель исследования
Построить модель с максимально большим значением `accuracy`. 


#### Задачи исследования
* довести долю правильных ответов до 0.75 и выше
* проверить `accuracy` на тестовой выборке

#### Описание данных
Каждый объект в наборе данных — это информация о поведении одного пользователя за месяц. Известно:

* `сalls` — количество звонков
* `minutes` — суммарная длительность звонков в минутах
* `messages` — количество sms-сообщений
* `mb_used` — израсходованный интернет-трафик в Мб
* `is_ultra` — каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0).

#### Используемые библиотеки

* pandas
* sklearn
* warnings

[назад к содержанию](#content)

<a class='anchor' id='get'></a>
## 2. Получение данных

Импортируем модули и библиотеки, которые потребуются в течение всего выполнения.

In [1]:
import pandas as pd
import warnings
import joblib

from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split as tts

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

Отключим предупреждения среды `Jupiter`.

In [2]:
warnings.filterwarnings('ignore')

Установим настроечный параметр библиотеки `pandas` для отображения всех столбцов при просмотре датафреймов.

In [3]:
pd.set_option("display.max_columns", None)

Загрузим подготовленный датафрейм, сохранив в переменной `df`, и изучим его.

In [4]:
# домашняя директория проекта на локальной машине под управлением Windows
PROJECT_DIR = r"C:\Users\e6ton\PycharmProjects\yandex-practicum"

try:
    if os.path.exists(PROJECT_DIR):
        df = pd.read_csv(os.path.join(PROJECT_DIR, 'datasets', 'users_behavior.csv'))
    else:
        df = pd.read_csv('/datasets/users_behavior.csv')
except:
    df = pd.read_csv('https://code.s3.yandex.net/datasets/users_behavior.csv')

In [5]:
df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [7]:
df.isna().sum().sum()

np.int64(0)

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df['is_ultra'].value_counts()

is_ultra
0    2229
1     985
Name: count, dtype: int64

**Промежуточный вывод**

Мы иммеем 3214 объектов за месяц с пятью признаками:

* `сalls` — количество звонков
* `minutes` — суммарная длительность звонков в минутах
* `messages` — количество sms-сообщений
* `mb_used` — израсходованный интернет-трафик в Мб
* `is_ultra` — каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0)

Это совокупоность абонентов за месяц, которые уже перешли на новые тарифы `'Ультра'` и `'Cмарт'` - это и есть наш целевой признак, который обозначен `1` и `0` с соотношением ~30% на ~70% соответственно.

Поскольку известны целевой и другие признаки, а категория тарифа выражена количественной переменной, но ограничена двумя значениями, мы будем решать задачу обучения с учителем с помощью бинарной классификации.

[назад к содержанию](#content)

<a class='anchor' id='research'></a>
## 3. Ход исследования

* [3.1. Разделяем данные на выборки](#3-1)
* [3.2. Исследуем модели](#3-2)
* [3.3. Проверяем лучшую модель на тестовой выборке](#3-3)
* [3.4. Проверяем лучшую модель на адекватность](#3-4)


[назад к содержанию](#content)

[_результаты исследования_](#result)

<a class='anchor' id='3-1'></a>
### 3.1. Разделяем данные на выборки

Так как спрятанной тестовой выборки нет (мы сами её формируем), то нам потребуются обучающая, валидационная и тестовая выборки. Разделим исходный датафрейм в соотношении 3:1:1, а перед этим выделим отдельно целевой и все остальные признаки.

In [10]:
features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

features_train, features_valid_test, target_train, target_valid_test = \
    tts(features, target, test_size=0.4, train_size=0.6, random_state=3, shuffle=True)


features_valid, features_test, target_valid, target_test = \
    tts(features_valid_test, target_valid_test, test_size=0.5, train_size=0.5, random_state=3, shuffle=True)

print(f"Результат разбивки в пропорции 3:1:1")
print(f"Тренировочная выборка {features_train.shape=}, {target_train.shape=}")
print(f"Валидационная выборка {features_valid.shape=}, {target_valid.shape=}")
print(f"Тестовая выборка {features_test.shape=}, {target_test.shape=}")

Результат разбивки в пропорции 3:1:1
Тренировочная выборка features_train.shape=(1928, 4), target_train.shape=(1928,)
Валидационная выборка features_valid.shape=(643, 4), target_valid.shape=(643,)
Тестовая выборка features_test.shape=(643, 4), target_test.shape=(643,)


**Промежуточный вывод**

Мы подготовили три случайные выборки из общей совокупности данных с сохранением пропорции распределения 70% нулей и 30% единиц (параметр `shuffle`):

* тренировочную размером 1928 объектов
* валидационную размером 643 объекта
* тестовую размером 643 объекта

[_назад к ходу исследования_](#research)

[назад к содержанию](#content)

<a class='anchor' id='3-2'></a>
### 3.2. Исследуем модели

Воспользуемся алгоритмом - классификатором решений `DecisionTreeClassifier`.

Для того, чтобы найти наилучший результат, будем менять гипермараметр `max_depth` (максимальная глубина дерева) алгоритма при создании модели в цикле, а правильность определять по метрике качества `accuracy`.

In [11]:
best_accuracy = 0

for depth in range(1, 6):
    model = DecisionTreeClassifier(random_state=3, max_depth=depth)
    model.fit(features_train, target_train)
    predictions_valid = model.predict(features_valid)
    curr_accuracy = accuracy_score(target_valid, predictions_valid)
    
    if curr_accuracy > best_accuracy:
        best_accuracy = curr_accuracy
    
print(f"Лучший результат с глубиной {depth} и {best_accuracy=}")

Лучший результат с глубиной 5 и best_accuracy=0.7947122861586314


Текущее значение правильности 79.4% с небольшим.

Проверим, сможем ли мы добиться лучшего значения метрики качества с другим алгоритмом - классификатором случайного леса (`RandomForestClassifier`). Заведём дополнительные переменные `best_est` (лучшее кол-во наблюдателей), `best_depth` (лучшая глубина дерева), `best_accuracy` и `best_model` - с целью повторного использования, если лучшая модель будет выявлена с помощью этого алгоритма.

In [12]:
best_est = 0
best_depth = 0
best_accuracy = 0
best_model = None

for est in range(10, 51, 5):
    for depth in range(1, 6):
        model = RandomForestClassifier(random_state=3, criterion='gini', n_estimators=est, max_depth=depth)
        model.fit(features_train, target_train)
        predictions_valid = model.predict(features_valid)
        curr_accuracy = accuracy_score(target_valid, predictions_valid)
        
        if curr_accuracy > best_accuracy:
            best_model = model
            best_est = est
            best_depth = depth
            best_accuracy = curr_accuracy

print(f"Наилуший показатель {best_accuracy=} в процессе построения деревьев найден с параметрами {best_est=} и {best_depth=}")

Наилуший показатель best_accuracy=0.8133748055987559 в процессе построения деревьев найден с параметрами best_est=35 и best_depth=5


Нам удалось добиться большего значения `best_accuracy`: было 79.4% при классификаторе решений `DecisionTreeClassifier`, а стало 81.3% с использованием алгоритма классификации случайного леса `RandomForestClassifier`, где 35 наблюдателей и 5 глубина.

Попробуем использовать алгоритм логистической регрессии. Будем менять количество итераций в цикле с шагом 100 от 1 до 1000.

Для этой цели также инициализируем две переменные `best_accuracy` и `best_qnt_iter` (лучшее количество итераций).

In [13]:
best_accuracy = 0
best_qnt_iter = 0

for qnt_iter in range(1, 1001, 100):
    model = LogisticRegression(random_state=3, solver='lbfgs', max_iter=qnt_iter)
    model.fit(features_train, target_train)
    predictions_valid = model.predict(features_valid)
    curr_accuracy = accuracy_score(target_valid, predictions_valid)
    
    if curr_accuracy > best_accuracy:
        best_accuracy = curr_accuracy
        best_qnt_iter = qnt_iter
        
print(f"Наилуший показатель {best_accuracy=} на логистической регрессии с количеством итераций {best_qnt_iter=}")

Наилуший показатель best_accuracy=0.7651632970451011 на логистической регрессии с количеством итераций best_qnt_iter=101


Логистическая регрессия показала наихудший результат.

**Промежуточный вывод**

Наилучшие показатели `accuracy` для разных алгоритмов:

* `DecisionTreeClassifier` c глубиной 5 >>> 79.4%
* `RandomForestClassifier` с 35 наблюдателями и глубиной 5 >>> 81.3%
* `LogisticRegression` c 101 итерацией >>> 76.5%

Таким образом, все три алгоритма решают нашу задачу **"Довести долю правильных ответов до 0.75 и выше"**, при этом `RandomForestClassifier` является лучшей. Модель с оптимальными гиперматаметрами сохранена в переменной `best_model`.

[_назад к ходу исследования_](#research)

[назад к содержанию](#content)

<a class='anchor' id='3-3'></a>
### 3.3. Проверяем лучшую модель на тестовой выборке

Самое время проверить предсказания модели на тестовой выборке, чтобы исключить наличие проблем переобучения или недообучения.

In [14]:
predictions_train = best_model.predict(features_train)
train_accuracy = accuracy_score(target_train, predictions_train)

predictions_test = best_model.predict(features_test)
test_accuracy = accuracy_score(target_test, predictions_test)

print(f"Accuracy для обучающей выборки {train_accuracy=}")
print(f"Accuracy для тестовой выборки {test_accuracy=}")

Accuracy для обучающей выборки train_accuracy=0.825207468879668
Accuracy для тестовой выборки test_accuracy=0.8087091757387247


Создадим два объекта со случайными признаками, которые отличаются друг от друга, чтобы выполнить ещё одну проверку.

In [15]:
new_features = pd.DataFrame(
    [
        [150.0, 901.22, 80.0, 36548.33], 
        [16.0, 141.99, 10.0, 12999.99]
    ],
    columns=features.columns)

answers = best_model.predict(new_features)

print(answers)

[1 0]


**Промежуточный вывод**

Наша модель не "вызубрила" ответы - погрешность состаляет 1.7% между обучающей и тестовой выборками.

[_назад к ходу исследования_](#research)

[назад к содержанию](#content)

<a class='anchor' id='3-4'></a>
### 3.4. Проверяем лучшую модель адекватность

Для проверки адекватности нашей лучшей модели используем функцию матрицы ошибок `confusion_matrix`.

In [16]:
cm = confusion_matrix(target_test, predictions_test)
print("Матрица ошибок:")
print(cm)

Матрица ошибок:
[[430  16]
 [107  90]]


Мы имеем:

* 430 истинно-положительных предсказания (тариф 'Ultra')
* 16 ложно-положительных предсказания (тариф 'Ultra')
* 107 ложно-отрицательных предсказания (тариф 'Smart')
* 90 истинно-отрицательных предсказания (тариф 'Smart')

Как видим, модель плохо предсказывает тариф `Smart` - основная совокупность ошибок здесь.

Попробуем воспользоваться константным классом `DummyClassifier` из библиотеки scikit-learn - такой алгоритм просто предсказывает наиболее частый класс (в нашем случае 0 - тариф `Smart`, 70% от всех данных).

In [17]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(features_train, target_train)

test_pred_dummy = dummy.predict(features_test)
test_pred_model = best_model.predict(features_test)

dummy_accuracy = accuracy_score(target_test, test_pred_dummy)
dummy_cm = confusion_matrix(target_test, test_pred_dummy)

# Сравнение результатов
print("Результаты базового классификатора:")
print(f"Точность: {dummy_accuracy:.2f}")
print("Матрица ошибок:")
print(dummy_cm)

Результаты базового классификатора:
Точность: 0.69
Матрица ошибок:
[[446   0]
 [197   0]]


Как видим, константный классификатор предсказал:

* 446 истинно-положительных предсказания (тариф 'Ultra')
* 197 ложно-отрицательных предсказания (тариф 'Smart')

Таким образом, 197 объектов с классом 0 (тариф 'Smart') были предсказаны как положительные 1 (тариф 'Ultra') - это неверно.
Accuracy составляет 69%, что ниже этой же метрики у нашей лучшей модели на 12,3%.

Сохраним лучшую модель

In [19]:
model_filename = 'recommendation_model.pkl'

# joblib.dump(best_model, model_filename)

**Промежуточный вывод**

Улучшить нашу лучшую модель можно через обогощение данных в объёме и/или в количестве признаков, однако на текущий момент это не представляется возможным.

[_назад к ходу исследования_](#research)

[назад к содержанию](#content)

<a class='anchor' id='result'></a>
## 4. Результаты исследования

В ходе исследования мы обучили три модели, подходящих для бинарной классификации, на тренировочной выборке с целью найти лучшую метрику правильности `accuracy` на валидационной выборке. Она определяет, насколько точно модель может предсказать подходящий абоненту тарифный план, чтобы повысить вероятность перехода с архивных тарифов на `Smart`или `Ultra`. Проверили предсказания лучшей модели на тестовой выборке, и на адекватность.

Результаты:

* `DecisionTreeClassifier` - 79.4%
* `RandomForestClassifier` - 81.3%
* `LogisticRegression` - 76.5%

* Модель сохранена в переменной `best_model` и может быть использована для рекомендаций с качеством предсказаний 81.3%
* Модель хорошо предсказывает тариф `Ultra` и средне тариф `Smart`

#### Задачи исследования

<div class="alert alert-info">
<b> 1. Довести долю правильных ответов до 0.75 и выше </div>

Доля правильных ответов составляет 0.813. 
    
Задача выполнена успешно ✓

<div class="alert alert-info">
<b> 2. Проверить accuracy (правльность предсказаний) на тестовой выборке </div>

* Accuracy для обучающей выборки составляет 0.825
* Accuracy для тестовой выборки составляет 0.808
    
Погрешность 0.017 в пределах нормы.
    
Задача выполнена успешно ✓

<div class="alert alert-info anchor" style="color:green" id="recommendation">
<b> Рекомендации </div>
    
Для повышения правильности предсказаний (метрики `accuracy`) требуются:
* данные за 2-ой месяц использования новых тарифов
* дополнительные признаки данных (например: домашний регион, выезд в роуминг, количество входящих звонков, способ оплаты и т.п.)
    
[назад к содержанию](#content)

<a class='anchor' id='check'></a>
## 5. Чек-лист готовности проекта

Поставьте 'x' в выполненных пунктах.

- [x] Jupyter Notebook открыт
- [x] Весь код исполняется без ошибок
- [x] Ячейки с кодом расположены в порядке исполнения
- [x] Выполнено задание 1: данные загружены и изучены
- [x] Выполнено задание 2: данные разбиты на три выборки
- [x] Выполнено задание 3: проведено исследование моделей
    - [x] Рассмотрено больше одной модели
    - [x] Рассмотрено хотя бы 3 значения гипепараметров для какой-нибудь модели
    - [x] Написаны выводы по результатам исследования
- [x] Выполнено задание 3: Проведено тестирование
- [x] Удалось достичь accuracy не меньше 0.75

[назад к содержанию](#content)